# 07b — Repetir o ajuste fino com taxa de aprendizado menor

Execute no **Colab com GPU T4**, em uma sessão nova. Reutilize o mesmo ZIP
HomeObjects + SUN RGB-D e o mesmo `best.pt` **antigo** já enviados ao Drive.

O experimento anterior parou na época 11 e seu melhor resultado foi na época 1.
Este experimento testa uma hipótese: reduzir **somente a taxa inicial de
aprendizado de 0.0003 para 0.00003**, dez vezes menor. O objetivo é fazer
mudanças menores nos pesos. Melhoria ainda precisa ser medida.

Os parâmetros de treinamento restantes seguem o 07: até 30 épocas, patience
10, batch 16, imagens 640, seed 42, AdamW e os mesmos aumentos de dados.
A agenda relativa de aprendizado é igual, com os valores reduzidos pela nova
taxa inicial. A avaliação registra também o baseline na validação mista e
compara o resultado com o primeiro ajuste fino, quando seus relatórios estiverem
na pasta anterior do Drive. Essas avaliações adicionais não mudam o treino.

A seleção usa validação. As 403 imagens TEST continuam fora da seleção e
do treinamento. A arquitetura, os 12 IDs e a política V2 seguem os existentes.


## 1. Instalar a versão definida e ativar a GPU

Usamos a versão do Ultralytics das comparações locais anteriores. Se o Colab pedir
reinício depois de instalar, reinicie e execute novamente a partir desta etapa.


In [1]:
%pip install -q ultralytics==8.4.150 pyyaml pandas pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 1.9 MB/s eta 0:00:00


In [2]:
import sys
import json
import hashlib
import platform
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import yaml
import torch
import ultralytics
from ultralytics import YOLO
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("No Colab, selecione Ambiente de execução → Alterar tipo → GPU T4 e reconecte.")
if ultralytics.__version__ != "8.4.150":
    raise RuntimeError("Reinicie o ambiente para carregar ultralytics==8.4.150.")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Python: 3.13.15
PyTorch: 2.11.0+cu130
Ultralytics: 8.4.150
GPU: Tesla T4


## 2. Montar o Drive e conferir as entradas

No Meu Drive, dentro de `TCC_Visao_Computacional`, use:

| Arquivo | Caminho |
|---|---|
| ZIP gerado no 06 | `datasets/homeobjects_sunrgbd_validado.zip` |
| Modelo antigo | `modelos/yolov8n_baseline/best.pt` |

Os resultados vão para uma pasta nova com data/hora. Não substituem o baseline.


In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
DRIVE_TCC = Path("/content/drive/MyDrive/TCC_Visao_Computacional")
ZIP_DATASET = DRIVE_TCC / "datasets/homeobjects_sunrgbd_validado.zip"
MODELO_ANTIGO_DRIVE = DRIVE_TCC / "modelos/yolov8n_baseline/best.pt"
SHA_MODELO_ANTIGO = "4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9"
NOME_EXPERIMENTO = "yolov8n_baseline_sunrgbd_lr_menor_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
PASTA_TRABALHO = Path("/content/tcc_sunrgbd") / NOME_EXPERIMENTO
PASTA_RESULTADOS = DRIVE_TCC / "resultados" / NOME_EXPERIMENTO
PASTA_RUN = PASTA_RESULTADOS / "treinamento"

# Mesmo experimento de dados; a mudança de treino é somente LR0.
EPOCHS = 30
PATIENCE = 10
BATCH = 16
IMGSZ = 640
SEED = 42
WORKERS = 2
LR0 = 0.00003

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with Path(caminho).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

for p in [ZIP_DATASET, MODELO_ANTIGO_DRIVE]:
    if not p.is_file():
        raise FileNotFoundError(f"Arquivo ausente no Drive: {p}")
if sha256_arquivo(MODELO_ANTIGO_DRIVE) != SHA_MODELO_ANTIGO:
    raise RuntimeError("O arquivo do Drive não corresponde ao modelo antigo avaliado. Copie o best.pt de models/yolov8n_baseline.")
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=False)
PASTA_TRABALHO.mkdir(parents=True, exist_ok=False)
MODELO_INICIAL = PASTA_TRABALHO / "baseline_inicial.pt"
shutil.copy2(MODELO_ANTIGO_DRIVE, MODELO_INICIAL)
print("Modelo inicial confirmado:", MODELO_INICIAL)
print("Resultados:", PASTA_RESULTADOS)

CONTROLE_07 = json.loads('{"experimento": "yolov8n_baseline_sunrgbd_20261013_120842_048419", "modelo_base_sha256": "4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9", "zip_sha256": "ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676", "test_sha256": "f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89", "politica_sha256": "1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845", "epochs": 30, "patience": 10, "batch": 16, "imgsz": 640, "seed": 42, "workers": 2, "optimizer": "AdamW", "lr0": 0.0003, "resume": false, "mosaic": 0.15, "close_mosaic": 5, "python": "3.13.15", "torch": "2.11.0+cu130", "ultralytics": "8.4.150", "gpu": "Tesla T4"}')
PASTA_ANTERIOR = DRIVE_TCC / "resultados" / CONTROLE_07["experimento"]
if sha256_arquivo(ZIP_DATASET) != CONTROLE_07["zip_sha256"]:
    raise RuntimeError("O ZIP é diferente do usado no 07. Para comparar a taxa, use o mesmo pacote.")
if LR0 != 0.00003:
    raise RuntimeError("Este experimento foi definido com LR0=0.00003.")
print("Experimento anterior:", PASTA_ANTERIOR)
print("Taxa de aprendizado:", CONTROLE_07["lr0"], "→", LR0)


Modelo inicial confirmado: /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209/baseline_inicial.pt
Resultados: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209
Experimento anterior: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline_sunrgbd_20261013_120842_048419
Taxa de aprendizado: 0.0003 → 3e-05


## 3. Extrair e conferir o pacote completo

Cada arquivo do manifesto é conferido por SHA-256. Também verificamos os 12 IDs,
as contagens e a ausência de imagens/labels adicionais fora do manifesto.
O YAML do Colab é uma cópia com a raiz local corrigida.


In [5]:
destino = PASTA_TRABALHO.resolve()
with zipfile.ZipFile(ZIP_DATASET) as pacote:
    nomes_zip = pacote.namelist()
    if len(nomes_zip) != len(set(nomes_zip)):
        raise ValueError("ZIP contém nomes repetidos.")
    for membro in pacote.infolist():
        final = (destino / membro.filename).resolve()
        if final != destino and destino not in final.parents:
            raise ValueError("ZIP contém caminho fora da pasta de trabalho.")
    pacote.extractall(destino)

manifesto = json.loads((PASTA_TRABALHO / "verificacao_preparacao.json").read_text(encoding="utf-8"))
for item in manifesto["arquivos"]:
    arquivo = (PASTA_TRABALHO / item["arquivo"]).resolve()
    if destino not in arquivo.parents or not arquivo.is_file() or sha256_arquivo(arquivo) != item["sha256"]:
        raise RuntimeError(f"Arquivo ausente ou alterado: {item['arquivo']}")
if manifesto["test"]["imagens"] != 403:
    raise RuntimeError("O pacote não preserva o TEST de 403 imagens.")
RAIZ_DATASET = PASTA_TRABALHO / manifesto["dataset_dir"]
dados_yaml = yaml.safe_load((RAIZ_DATASET / "data.yaml").read_text(encoding="utf-8-sig"))
NOMES = {int(k): str(v) for k, v in dados_yaml["names"].items()}
ESPERADAS = dict(enumerate(["bed", "sofa", "chair", "table", "lamp", "tv", "laptop", "wardrobe", "window", "door", "potted plant", "photo frame"]))
if NOMES != ESPERADAS or NOMES != {int(k): v for k, v in manifesto["classes"].items()}:
    raise RuntimeError("Classes/IDs incompatíveis com o baseline.")
verificacao = []
for split in ["train", "val", "test"]:
    for tipo in ["images", "labels"]:
        prefixo = f"{manifesto['dataset_dir']}/{tipo}/{split}/"
        esperados = {i["arquivo"] for i in manifesto["arquivos"] if i["arquivo"].startswith(prefixo)}
        reais = {p.relative_to(PASTA_TRABALHO).as_posix() for p in (RAIZ_DATASET / tipo / split).rglob("*") if p.is_file()}
        if reais != esperados or len(reais) != manifesto["imagens_por_split"][split]:
            raise RuntimeError(f"Arquivos adicionais/ausentes ou contagem diferente: {tipo}/{split}.")
    verificacao.append({"split": split, "imagens": manifesto["imagens_por_split"][split]})
dados_yaml["path"] = RAIZ_DATASET.as_posix()
YAML_MISTO = RAIZ_DATASET / "data_colab.yaml"
YAML_MISTO.write_text(yaml.safe_dump(dados_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")
sys.path.insert(0, str(PASTA_TRABALHO / "codigo"))
from src.validacao_alertas import carregar_home_val, avaliar_alertas
import src.politica_alertas as politica
REGISTROS_HOME_VAL = carregar_home_val(RAIZ_DATASET, NOMES)
lista_home_val = PASTA_TRABALHO / "home_val.txt"
lista_home_val.write_text("\n".join(str(r["imagem"]) for r in REGISTROS_HOME_VAL) + "\n", encoding="utf-8")
# Uma avaliação de detecção somente na validação Home, separada da validação mista.
YAML_HOME_VAL = PASTA_TRABALHO / "data_home_val.yaml"
home_yaml = {**dados_yaml, "val": str(lista_home_val)}
YAML_HOME_VAL.write_text(yaml.safe_dump(home_yaml, allow_unicode=True, sort_keys=False), encoding="utf-8")
display(pd.DataFrame(verificacao))
print("Validação Home:", len(REGISTROS_HOME_VAL), "imagens")
shutil.copy2(PASTA_TRABALHO / "verificacao_preparacao.json", PASTA_RESULTADOS / "verificacao_preparacao.json")
shutil.copy2(PASTA_TRABALHO / "resumo_preparacao_sunrgbd.json", PASTA_RESULTADOS / "resumo_preparacao_sunrgbd.json")


,split,imagens
0,train,3970
1,val,993
2,test,403


Validação Home: 456 imagens


PosixPath('/content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209/resumo_preparacao_sunrgbd.json')

## 4. Registrar configuração e avaliar o antigo na validação Home

O controle e o candidato usam o mesmo conjunto, GPU e parâmetros. As métricas
dos alertas usam confiança 0,25, NMS IoU 0,70, correspondência de mesma classe
com IoU ≥ 0,50 e alertas de prioridade média/alta. As métricas mAP usam 0,001.


Também avaliamos o antigo na **validação mista de 993 imagens**, separadamente
das 456 imagens HomeObjects. Isso fornece o ponto de partida da curva de
treinamento. Não compare diretamente métricas obtidas em conjuntos diferentes.


In [6]:
CONFIGURACAO = {
    "experimento": NOME_EXPERIMENTO, "modelo_base_sha256": SHA_MODELO_ANTIGO,
    "zip_sha256": sha256_arquivo(ZIP_DATASET), "epochs": EPOCHS,
    "patience": PATIENCE, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED,
    "workers": WORKERS, "optimizer": "AdamW", "lr0": LR0,
    "resume": False, "mosaic": 0.15, "close_mosaic": 5,
    "python": platform.python_version(), "ultralytics": ultralytics.__version__,
    "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
    "test_sha256": manifesto["test"]["sha256"],
    "politica_sha256": sha256_arquivo(politica.__file__),
    "home_val_imagens": len(REGISTROS_HOME_VAL),
    "criterio_checkpoint": "best.pt por fitness de validação mista do Ultralytics",
    "criterio_triagem": "ganho de precisão de alertas com Recall preservado, na validação Home",
}
CONFIGURACAO["experimento_controle"] = CONTROLE_07["experimento"]
CONFIGURACAO["alteracao_planejada"] = {"lr0": {"antes": 0.0003, "agora": LR0}}
for chave in ["modelo_base_sha256", "zip_sha256", "test_sha256", "politica_sha256",
              "epochs", "patience", "batch", "imgsz", "seed", "workers", "optimizer",
              "resume", "mosaic", "close_mosaic"]:
    if CONFIGURACAO[chave] != CONTROLE_07[chave]:
        raise RuntimeError(f"Outro parâmetro de comparação mudou: {chave}")
diferencas_ambiente = {k: {"antes": CONTROLE_07[k], "agora": CONFIGURACAO[k]}
                      for k in ["python", "torch", "ultralytics", "gpu"]
                      if CONTROLE_07[k] != CONFIGURACAO[k]}
CONFIGURACAO["diferencas_ambiente"] = diferencas_ambiente
if diferencas_ambiente:
    print("Ambiente mudou; considere isso na comparação:", diferencas_ambiente)
(PASTA_RESULTADOS / "configuracao_experimento.json").write_text(
    json.dumps(CONFIGURACAO, ensure_ascii=False, indent=2), encoding="utf-8")

def avaliar_detector_val(modelo, nome, caminho_yaml=YAML_HOME_VAL):
    r = modelo.val(data=str(caminho_yaml), split="val", imgsz=IMGSZ,
                   batch=BATCH, device=0, conf=0.001, iou=0.70, max_det=300,
                   plots=True, project=str(PASTA_RESULTADOS), name=nome, exist_ok=False)
    return {"precision_detector": float(r.box.mp), "recall_detector": float(r.box.mr),
            "map50": float(r.box.map50), "map50_95": float(r.box.map)}

antigo = YOLO(str(MODELO_INICIAL))
if {int(k): str(v) for k, v in antigo.names.items()} != NOMES:
    raise RuntimeError("Modelo antigo tem IDs diferentes dos labels.")
metricas_antigo = {**avaliar_detector_val(antigo, "baseline_home_val"),
                   **avaliar_alertas(antigo, REGISTROS_HOME_VAL, NOMES)}
(PASTA_RESULTADOS / "metricas_baseline_val.json").write_text(json.dumps(metricas_antigo, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metricas_antigo, ensure_ascii=False, indent=2))
metricas_baseline_misto = avaliar_detector_val(antigo, "baseline_misto_val", YAML_MISTO)
(PASTA_RESULTADOS / "metricas_baseline_misto_val.json").write_text(
    json.dumps(metricas_baseline_misto, ensure_ascii=False, indent=2), encoding="utf-8")
print("Ponto de partida na validação mista:", metricas_baseline_misto)
del antigo
torch.cuda.empty_cache()


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1724.8±391.5 MB/s, size: 141.4 KB)
val: Scanning /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209/homeobjects_yolo/labels/val/raw_train... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 2.2Kit/s 0.2s
val: New cache created: /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209/homeobjects_yolo/labels/val/raw_train.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.1it/s 9.3s
                   all        456       3682      0.883      0.765      0.859      0.664
                   bed         29         29      0.907      0.793      0.879      0.719
                  sofa        294        420      0.912      0.921      0.952      0.792


## 5. Ajuste fino com LR0=0.00003

Inicializamos novamente os pesos do modelo antigo. A taxa inicial é dez vezes
menor. As 30 épocas e patience 10 seguem iguais ao primeiro experimento.
A parada antecipada continua sendo uma possibilidade; o número de épocas
concluídas não determina sozinho a qualidade do resultado.

Os pesos e logs vão para uma **nova pasta** do Drive. O arquivo `best.pt` segue
o critério padrão do Ultralytics na validação mista; os alertas são comparados
depois na mesma validação HomeObjects. Os dados do TEST não escolhem o candidato.


In [7]:
modelo = YOLO(str(MODELO_INICIAL))
resultado_treino = modelo.train(
    data=str(YAML_MISTO), epochs=EPOCHS, patience=PATIENCE,
    imgsz=IMGSZ, batch=BATCH, device=0, workers=WORKERS,
    seed=SEED, deterministic=True, optimizer="AdamW", lr0=LR0,
    lrf=0.10, cos_lr=True, warmup_epochs=2.0, warmup_bias_lr=0.0,
    weight_decay=0.0005, mosaic=0.15, close_mosaic=5,
    mixup=0.0, scale=0.20, translate=0.10, fliplr=0.50,
    resume=False, pretrained=True, save=True, save_period=5,
    plots=True, cache=False, project=str(PASTA_RESULTADOS),
    name="treinamento", exist_ok=False,
)
PASTA_RUN = Path(modelo.trainer.save_dir)
BEST = PASTA_RUN / "weights/best.pt"
if not BEST.is_file():
    raise FileNotFoundError(f"best.pt não produzido: {BEST}")
shutil.copy2(BEST, PASTA_RESULTADOS / "best.pt")
print("Novo candidato preservado:", PASTA_RESULTADOS / "best.pt")
del modelo
torch.cuda.empty_cache()


New https://pypi.org/project/ultralytics/8.4.176 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209/homeobjects_yolo/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0

## 6. Comparar com o antigo na mesma validação

O `best.pt` é escolhido pelo critério padrão do Ultralytics na validação mista;
não é necessariamente a época de maior precisão dos alertas. A comparação
abaixo verifica se ele atende ao objetivo do projeto. Não aumente a confiança
com base no TEST para fazer a precisão parecer maior.


In [8]:
candidato = YOLO(str(PASTA_RESULTADOS / "best.pt"))
metricas_candidato = {**avaliar_detector_val(candidato, "candidato_home_val"),
                      **avaliar_alertas(candidato, REGISTROS_HOME_VAL, NOMES)}
(PASTA_RESULTADOS / "metricas_candidato_val.json").write_text(json.dumps(metricas_candidato, indent=2, ensure_ascii=False), encoding="utf-8")
chaves = ["precision_detector", "recall_detector", "map50", "map50_95",
          "precision_alertas", "recall_alertas", "f1_alertas", "fn", "fp", "alta_sem_alerta"]
comparacao = pd.DataFrame([
    {"metrica": k, "antigo": metricas_antigo[k], "candidato": metricas_candidato[k],
     "diferenca": None if metricas_antigo[k] is None or metricas_candidato[k] is None else metricas_candidato[k] - metricas_antigo[k]}
    for k in chaves
])
display(comparacao)
comparacao.to_csv(PASTA_RESULTADOS / "comparacao_validacao.csv", index=False, encoding="utf-8-sig")
pa, pn = metricas_antigo["precision_alertas"], metricas_candidato["precision_alertas"]
ra, rn = metricas_antigo["recall_alertas"], metricas_candidato["recall_alertas"]
promissor = all(v is not None for v in [pa, pn, ra, rn]) and pn > pa and rn >= ra and metricas_candidato["alta_sem_alerta"] <= metricas_antigo["alta_sem_alerta"]
triagem = {"promissor_na_validacao": bool(promissor),
           "criterio": "Precisão dos alertas maior, Recall não menor e omissões de alta prioridade não maiores, em Home VAL.",
           "observacao": "Triagem exploratória; não é teste de significância nem validação de risco real. Histórico do baseline pode incluir parte do Home VAL atual.",
           "proximo_passo": "Comparar antigo e candidato no 05d local, com TEST e parâmetros preservados; não substituir o baseline automaticamente."}
(PASTA_RESULTADOS / "triagem_validacao.json").write_text(json.dumps(triagem, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(triagem, ensure_ascii=False, indent=2))
print("Resultados:", PASTA_RESULTADOS)


# Registra o desempenho do candidato no mesmo conjunto que selecionou a época.
metricas_candidato_misto = avaliar_detector_val(candidato, "candidato_misto_val", YAML_MISTO)
(PASTA_RESULTADOS / "metricas_candidato_misto_val.json").write_text(
    json.dumps(metricas_candidato_misto, ensure_ascii=False, indent=2), encoding="utf-8")

# Comparação adicional com o ajuste anterior (0.0003), sem usar seu best.pt
# para inicializar o treinamento e sem misturar suas métricas com o TEST.
arquivo_controle = PASTA_ANTERIOR / "comparacao_validacao.csv"
if arquivo_controle.is_file():
    primeira = pd.read_csv(arquivo_controle)
    if primeira["metrica"].duplicated().any():
        raise RuntimeError("Relatório anterior tem métricas duplicadas.")
    referencia = primeira.set_index("metrica")
    linhas = []
    for k in chaves:
        if k not in referencia.index:
            raise RuntimeError(f"Métrica ausente no relatório anterior: {k}")
        antes = float(referencia.loc[k, "candidato"])
        agora = metricas_candidato[k]
        linhas.append({"metrica": k, "antigo_nesta_execucao": metricas_antigo[k],
                       "primeiro_ajuste_lr_0003": antes,
                       "novo_ajuste_lr_00003": agora,
                       "diferenca_entre_ajustes": None if agora is None else agora - antes})
    evolucao = pd.DataFrame(linhas)
    evolucao.to_csv(PASTA_RESULTADOS / "comparacao_tres_modelos_validacao.csv",
                   index=False, encoding="utf-8-sig")
    display(evolucao)
else:
    print("Relatório anterior não localizado; a comparação principal com o antigo foi salva normalmente.")


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2322.6±816.2 MB/s, size: 163.5 KB)
val: Scanning /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209/homeobjects_yolo/labels/val/raw_train... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 2.0Kit/s 0.2s
val: New cache created: /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209/homeobjects_yolo/labels/val/raw_train.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.8it/s 10.4s
                   all        456       3682      0.865       0.73      0.837      0.637
                   bed         29         29      0.916      0.759       0.83      0.708
                  sofa        294        420      0.898      0.912      0.945      0.766

,metrica,antigo,candidato,diferenca
0,precision_detector,0.882927,0.865007,-0.017920
1,recall_detector,0.765052,0.729839,-0.035213
2,map50,0.859207,0.837320,-0.021888
3,map50_95,0.664002,0.637415,-0.026588
4,precision_alertas,0.783646,0.771870,-0.011776
5,recall_alertas,0.899023,0.879479,-0.019544
6,f1_alertas,0.837379,0.822168,-0.015211
7,fn,155.000000,185.000000,30.000000
8,fp,381.000000,399.000000,18.000000
9,alta_sem_alerta,24.000000,25.000000,1.000000


{
  "promissor_na_validacao": false,
  "criterio": "Precisão dos alertas maior, Recall não menor e omissões de alta prioridade não maiores, em Home VAL.",
  "observacao": "Triagem exploratória; não é teste de significância nem validação de risco real. Histórico do baseline pode incluir parte do Home VAL atual.",
  "proximo_passo": "Comparar antigo e candidato no 05d local, com TEST e parâmetros preservados; não substituir o baseline automaticamente."
}
Resultados: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3590.1±1152.6 MB/s, size: 170.1 KB)
val: Scanning /content/tcc_sunrgbd/yolov8n_baseline_sunrgbd_lr_menor_20261013_135714_168209/homeobjects_yolo/labels/val/raw_train... 993 images, 0 backgrounds, 0 corr

,metrica,antigo_nesta_execucao,primeiro_ajuste_lr_0003,novo_ajuste_lr_00003,diferenca_entre_ajustes
0,precision_detector,0.882927,0.800123,0.865007,0.064884
1,recall_detector,0.765052,0.735975,0.729839,-0.006136
2,map50,0.859207,0.819436,0.837320,0.017884
3,map50_95,0.664002,0.616436,0.637415,0.020978
4,precision_alertas,0.783646,0.746674,0.771870,0.025196
5,recall_alertas,0.899023,0.877524,0.879479,0.001954
6,f1_alertas,0.837379,0.806828,0.822168,0.015340
7,fn,155.000000,188.000000,185.000000,-3.000000
8,fp,381.000000,457.000000,399.000000,-58.000000
9,alta_sem_alerta,24.000000,28.000000,25.000000,-3.000000


## 7. Enviar resultados e escolher o próximo passo

Arquivos para documentar o experimento 07b, na pasta de resultados do Drive:

- `configuracao_experimento.json`;
- `comparacao_validacao.csv`;
- `triagem_validacao.json`;
- `treinamento/results.csv` (a curva completa de épocas);
- `comparacao_tres_modelos_validacao.csv`, quando produzido;
- `metricas_baseline_misto_val.json` e `metricas_candidato_misto_val.json`.

Se houver ganho na validação, avaliamos o candidato no 05d local. Para esse
passo futuro, o caminho separado será `models/yolov8n_baseline_sunrgbd_lr_menor_01/best.pt`.
O antigo permanece como referência atual. Uma melhora apenas em relação
ao primeiro ajuste ainda não demonstra que o candidato superou o antigo.

O TEST já foi usado em comparações anteriores. Ele segue como benchmark fixo;
a escolha de taxa, classes e dados usa apenas desenvolvimento e validação.
A comparação Home VAL é exploratória, pois o baseline pode ter visto parte
desse conjunto no seu treinamento original. Não é validação independente
de risco físico. A política V2 produz os mesmos tipos de mensagens em texto.
